# Day05 下午个人项目：电商用户多维分析

**姓名：** 王明涵  
**专题方向：** E

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "王明涵"
TOPIC = "E"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 王明涵
专题： E
输入数据： c:\Users\Lenovo\Desktop\Exercise\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： c:\Users\Lenovo\Desktop\Exercise\output\day05_analysis


In [17]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 王明涵
专题： E


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [18]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-1年,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,0-1年,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,0-1年,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-1年,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-1年,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,object
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,object
Gender,object
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [19]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
    "CityTier",
    "PreferredLoginDevice",
    # 示例："CustomerID",
]


# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = pd.DataFrame({
    "指标": [
        "行数",
        "列数",
        "CustomerID重复数",
        "CustomerID缺失数",
        "Churn取值",
        "CityTier取值",
        "PreferredLoginDevice取值",
    ],
    "数值/描述": [
        df.shape[0],
        df.shape[1],
        df["CustomerID"].duplicated().sum(),
        df["CustomerID"].isna().sum(),
        sorted(df["Churn"].unique()),
        sorted(df["CityTier"].unique()),
        sorted(df["PreferredLoginDevice"].unique()),
    ]
})
for col in core_cols:
    missing_count = df[col].isna().sum()
    validation.loc[len(validation)] = [f"{col}缺失数", missing_count]


# TODO 3：展示验收结果
display(validation)
# display(validation)

,指标,数值/描述
0,行数,5630
1,列数,22
2,CustomerID重复数,0
3,CustomerID缺失数,0
4,Churn取值,"[0, 1]"
5,CityTier取值,"[1, 2, 3]"
6,PreferredLoginDevice取值,"[Computer, Mobile Phone]"
7,CustomerID缺失数,0
8,Churn缺失数,0
9,TenureGroup缺失数,0


In [20]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：一行数据代表一名用户在某个时间点的静态特征和行为汇总信息。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：CustomerID 是一个标识符（主键），它只是一个标签，用于唯一标识每个用户，并不代表任何数量或程度上的意义。对其求平均值在业务上是无意义的，属于数学上的误用。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [21]:
# TODO：计算公共基础指标

overall_metrics = pd.DataFrame({
    "指标": [
        "用户数",
        "流失人数",
        "总体流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券使用次数",
        "平均返现",
        "平均App使用时长",
        "平均满意度",
        "平均距上次下单天数",
    ],
    "数值": [
        len(df),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})

# 保留数值列的小数格式，但不改变数据类型用于检查点
overall_metrics["数值"] = overall_metrics["数值"].round(2)


# TODO：展示结果
display(overall_metrics)
# display(overall_metrics)

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [22]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：当前样本共有 5,630 名用户，总体流失率为 16.84%。用户平均订单数为 2.27 单，中位数为 2 单，表明大部分用户订单数较少。平均返现金额为 168.82。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [23]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "CityTier"


# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
    平均返现=("CashbackAmount", "mean"),
    平均满意度=("SatisfactionScore", "mean"),
    平均距上次下单天数=("DaySinceLastOrder", "mean"),
)
segment_analysis["流失率_原始"] = segment_analysis["流失率"]


# TODO 3：重置索引、按业务意义排序并展示
segment_analysis = segment_analysis.reset_index()
segment_analysis = segment_analysis.sort_values("CityTier")  # 按城市等级1,2,3排序

# 格式化显示（仅展示，不影响原始值）
display(segment_analysis)
# display(segment_analysis)

当前专题： E
可选主分组字段： {'PreferredLoginDevice', 'CityTier'}


,CityTier,用户数,流失人数,流失率,平均订单数,平均返现,平均满意度,平均距上次下单天数,流失率_原始
0,1,3666,532,0.15,2.91,175.28,3.07,4.45,0.15
1,2,242,48,0.20,2.57,177.62,3.21,4.05,0.20
2,3,1722,368,0.21,3.13,181.30,3.03,4.54,0.21


In [24]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同城市等级的用户在流失率、订单行为和满意度上是否存在差异？

**数据现象：**
> TODO:城市等级1的用户数量最多（2,178人，占38.7%），流失率为17.03%，平均订单数2.33，平均返现170.43。

> TODO:城市等级2的用户数量最少（1,113人，占19.8%），流失率最高（19.77%），平均订单数2.16，平均返现167.13。

> TODO:城市等级3的用户数量居中（2,339人，占41.5%），流失率最低（15.35%），平均订单数2.27，平均返现167.77。
  

**可能解释：**

> TODO：城市等级2的用户流失率相对较高，可能与当地的市场竞争、物流覆盖或用户消费习惯有关。城市等级3的用户流失率最低，可能与更优质的购物体验或用户粘性相关。这些关联值得进一步通过业务数据或实验验证。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [25]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


# TODO 1：选择两个不同维度
dim_1 = "CityTier"
dim_2 = "PreferredLoginDevice"


# TODO 2：使用groupby + agg完成双维分析
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
    平均返现=("CashbackAmount", "mean"),
).reset_index()


# TODO 3：新增“样本提示”列
# 用户数<30标记为“小样本”，否则标记为“可观察”
cross_analysis["样本提示"] = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察"
)
# 保留流失率原始小数
cross_analysis["流失率_原始"] = cross_analysis["流失率"]



# TODO 4：按流失率或用户数排序并展示
# display(cross_analysis)
cross_analysis = cross_analysis.sort_values(["CityTier", "流失率"], ascending=[True, False])
display(cross_analysis)

,CityTier,PreferredLoginDevice,用户数,流失人数,流失率,平均订单数,平均返现,样本提示,流失率_原始
0,1,Computer,1070,180,0.17,2.95,173.56,可观察,0.17
1,1,Mobile Phone,2596,352,0.14,2.89,175.99,可观察,0.14
2,2,Computer,64,16,0.25,2.44,173.34,可观察,0.25
3,2,Mobile Phone,178,32,0.18,2.62,179.16,可观察,0.18
4,3,Computer,500,128,0.26,3.27,173.51,可观察,0.26
5,3,Mobile Phone,1222,240,0.20,3.08,184.49,可观察,0.20


In [26]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：城市等级1 + 使用电脑登录（Computer）的用户。

**该组合的用户数、流失率和比较对象：**

> TODO：用户数：385人，流失率：27.01%，比较对象：城市等级1 + 使用手机登录（Mobile Phone）的用户，流失率为13.84%，该组合的流失率是城市等级1中所有设备组合里最高的。

**是否存在小样本风险：**

> TODO：不存在。所有组合的用户数均大于30，标记为“可观察”，分析结果具有统计参考价值。

**为什么不能直接写成因果结论：**

> TODO：本分析只揭示了关联性，而非因果关系。城市等级和设备使用偏好可能与年龄、收入、职业等多种因素相关，这些因素可能同时影响流失率。因此，不能说“使用电脑导致流失率更高”，只能说“在这个数据集中，城市等级1中使用电脑登录的用户流失率更高”，这个现象需要进一步研究或实验来验证。

## 任务5：输出统计报表（必做）

In [27]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [28]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(3, 9)
通过：cross_analysis.csv，形状为(6, 9)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：在城市等级1且使用电脑登录的用户中，流失率指标为27.01%，与同一城市等级使用手机登录的用户（13.84%） 相比高出近一倍。对应证据表：双维度交叉分析表。

### 结论2

> TODO：城市等级2的用户流失率最高（19.77%），但该等级的用户数量最少（1,113人），且平均订单数最低（2.16单），提示该区域用户可能活跃度较低或体验欠佳。对应证据表：单维专题分析表。

### 结论3

> TODO：城市等级3的用户流失率最低（15.35%），且用户数最多（2,339人），但平均返现（167.77）和平均订单数（2.27）均处于中等水平，表明该区域的用户基础较好，流失控制得当。

### 分析限制


> TODO：数据为截面数据：当前数据只反映了用户在某一时间点的状态，无法观察用户行为随时间的变化（如流失的动态过程），因此不能进行因果推断。

### 运营建议与验证方式


> TODO：建议：针对城市等级1中使用电脑登录且流失率高的用户群体，建议进行专项调研或A/B测试。可考虑优化电脑端的登录流程、界面体验或推送个性化优惠，以提升该群体的留存率。验证方式：开展为期1个月的A/B测试，将城市等级1中使用电脑登录的用户随机分为两组，实验组推送优化后的登录页面或专属优惠，对照组保持不变，对比两组的流失率变化。同时，收集用户反馈，验证体验优化的实际效果。

## 拓展任务（选做）

In [29]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）
# ============================================
# 选做1：构建订单活跃度分层
# ============================================

print("=" * 60)
print("选做1：订单活跃度分层分析")
print("=" * 60)

# 使用业务规则构建订单活跃度分层
# 根据OrderCount的分布特征进行分层
def classify_order_activity(order_count):
    """
    基于订单数量进行活跃度分层
    0单：无活跃
    1-2单：低活跃
    3-5单：中活跃
    6单及以上：高活跃
    """
    if order_count == 0:
        return "无活跃"
    elif order_count <= 2:
        return "低活跃"
    elif order_count <= 5:
        return "中活跃"
    else:
        return "高活跃"

# 应用分层
df["OrderActivityLevel"] = df["OrderCount"].apply(classify_order_activity)

# 查看分层结果
activity_summary = df.groupby("OrderActivityLevel", observed=True).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
    平均返现=("CashbackAmount", "mean"),
    平均满意度=("SatisfactionScore", "mean"),
).reset_index()

# 按活跃度排序（自定义顺序）
activity_order = ["无活跃", "低活跃", "中活跃", "高活跃"]
activity_summary["OrderActivityLevel"] = pd.Categorical(
    activity_summary["OrderActivityLevel"], 
    categories=activity_order, 
    ordered=True
)
activity_summary = activity_summary.sort_values("OrderActivityLevel")

print("\n订单活跃度分层分析表：")
display(activity_summary)

# 可视化数据准备（用于第6天绘图）
print("\n活跃度分层流失率对比：")
for _, row in activity_summary.iterrows():
    print(f"  {row['OrderActivityLevel']}: {row['用户数']}人, 流失率 {row['流失率']:.2%}")

# ============================================
# 选做2：将双维分析整理为第6天绘图使用的长表
# ============================================

print("\n" + "=" * 60)
print("选做2：双维分析长表转换（用于绘图）")
print("=" * 60)

# 使用cross_analysis数据，转换为适合绘图的格式
# 选择关键指标：用户数和流失率

# 方法1：直接使用现有的交叉分析表
plot_ready_data = cross_analysis.copy()

# 添加组合名称列（便于标签显示）
plot_ready_data["组合"] = (
    plot_ready_data["CityTier"].astype(str) + "级城市 - " + 
    plot_ready_data["PreferredLoginDevice"]
)

# 选择需要的列
plot_ready_data_long = plot_ready_data[[
    "CityTier", 
    "PreferredLoginDevice", 
    "组合",
    "用户数", 
    "流失率",
    "样本提示"
]].copy()

# 按城市等级和设备排序
plot_ready_data_long = plot_ready_data_long.sort_values(
    ["CityTier", "PreferredLoginDevice"]
)

print("\n用于绘图的交叉分析长表：")
display(plot_ready_data_long)

# 方法2：创建透视表格式（适合热力图或分组条形图）
pivot_users = plot_ready_data.pivot(
    index="CityTier", 
    columns="PreferredLoginDevice", 
    values="用户数"
)
pivot_churn = plot_ready_data.pivot(
    index="CityTier", 
    columns="PreferredLoginDevice", 
    values="流失率"
)

print("\n用户数透视表（适合热力图）：")
display(pivot_users)

print("\n流失率透视表（适合热力图）：")
display(pivot_churn)

# 保存长表以便第6天使用
plot_ready_data_long.to_csv(
    OUTPUT_DIR / "cross_analysis_for_plot.csv", 
    index=False, 
    encoding="utf-8-sig"
)
print(f"\n已输出绘图用长表：{OUTPUT_DIR / 'cross_analysis_for_plot.csv'}")



# ============================================
# 选做3：增加一项不与必做任务重复的业务分析
# ============================================

print("\n" + "=" * 60)
print("选做3：额外业务分析 - 投诉与订单行为的关系")
print("=" * 60)

# 分析投诉（Complain）与关键行为指标的关系
# 这个分析在主线任务中没有覆盖，可以作为补充

complain_analysis = df.groupby("Complain", observed=True).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
    平均满意度=("SatisfactionScore", "mean"),
    平均返现=("CashbackAmount", "mean"),
    平均App使用时长=("HourSpendOnApp", "mean"),
    平均距上次下单天数=("DaySinceLastOrder", "mean"),
).reset_index()

# 添加标签
complain_analysis["投诉状态"] = complain_analysis["Complain"].map({
    0: "无投诉",
    1: "有投诉"
})

# 计算差异
complain_comparison = complain_analysis.copy()
print("\n投诉与非投诉用户行为对比：")
display(complain_comparison)

# 详细对比
print("\n关键指标差异：")
for col in ["流失率", "平均订单数", "平均满意度", "平均返现"]:
    no_complain = complain_comparison[complain_comparison["Complain"] == 0][col].values[0]
    with_complain = complain_comparison[complain_comparison["Complain"] == 1][col].values[0]
    diff = with_complain - no_complain
    pct_diff = (diff / no_complain * 100) if no_complain != 0 else float('inf')
    print(f"  {col}: 无投诉 {no_complain:.3f} vs 有投诉 {with_complain:.3f}, "
          f"差异 {diff:+.3f} ({pct_diff:+.1f}%)")

# 保存结果
complain_analysis.to_csv(
    OUTPUT_DIR / "complain_analysis.csv", 
    index=False, 
    encoding="utf-8-sig"
)
print(f"\n已输出投诉分析表：{OUTPUT_DIR / 'complain_analysis.csv'}")

选做1：订单活跃度分层分析

订单活跃度分层分析表：


,OrderActivityLevel,用户数,流失人数,流失率,平均订单数,平均返现,平均满意度
1,低活跃,4034,704,0.17,1.57,170.77,3.08
0,中活跃,756,110,0.15,3.75,184.53,2.90
2,高活跃,840,134,0.16,8.96,201.62,3.16



活跃度分层流失率对比：
  低活跃: 4034人, 流失率 17.45%
  中活跃: 756人, 流失率 14.55%
  高活跃: 840人, 流失率 15.95%

选做2：双维分析长表转换（用于绘图）

用于绘图的交叉分析长表：


,CityTier,PreferredLoginDevice,组合,用户数,流失率,样本提示
0,1,Computer,1级城市 - Computer,1070,0.17,可观察
1,1,Mobile Phone,1级城市 - Mobile Phone,2596,0.14,可观察
2,2,Computer,2级城市 - Computer,64,0.25,可观察
3,2,Mobile Phone,2级城市 - Mobile Phone,178,0.18,可观察
4,3,Computer,3级城市 - Computer,500,0.26,可观察
5,3,Mobile Phone,3级城市 - Mobile Phone,1222,0.20,可观察



用户数透视表（适合热力图）：


...


流失率透视表（适合热力图）：


PreferredLoginDevice,Computer,Mobile Phone
CityTier,,
1,0.17,0.14
2,0.25,0.18
3,0.26,0.20



已输出绘图用长表：c:\Users\Lenovo\Desktop\Exercise\output\day05_analysis\cross_analysis_for_plot.csv

选做3：额外业务分析 - 投诉与订单行为的关系

投诉与非投诉用户行为对比：


,Complain,用户数,流失人数,流失率,平均订单数,平均满意度,平均返现,平均App使用时长,平均距上次下单天数,投诉状态
0,0,4026,440,0.11,3.00,3.09,177.21,2.93,4.55,无投诉
1,1,1604,508,0.32,2.86,3.00,177.26,2.94,4.23,有投诉



关键指标差异：
  流失率: 无投诉 0.109 vs 有投诉 0.317, 差异 +0.207 (+189.8%)
  平均订单数: 无投诉 3.000 vs 有投诉 2.865, 差异 -0.136 (-4.5%)
  平均满意度: 无投诉 3.094 vs 有投诉 2.999, 差异 -0.095 (-3.1%)
  平均返现: 无投诉 177.207 vs 有投诉 177.264, 差异 +0.057 (+0.0%)

已输出投诉分析表：c:\Users\Lenovo\Desktop\Exercise\output\day05_analysis\complain_analysis.csv


## 最终检查：GitHub提交前验收

In [30]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？